# 09 | Simulador da aplicação — somente HTTP

POC semissintética • contrato provisório • sem homologação acadêmica.
Execute em kernel reiniciado, de cima para baixo. As credenciais vêm do ambiente.


**Objetivo:** consumir a proposta persistida sem SQL, treino ou SDK de storage. **Entradas:** contexto salvo em08; override explícito opcional. **Esperado:** paginação completa, hashes e schema válidos.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.application_client import call, context, save_context

from src.workbench.application_client import select_proposal,fetch,table,explorer
PROPOSAL_ID=None  # Opcional: UUID escolhido na lista abaixo; contexto08 é automático.
display(pd.DataFrame(call('GET','/v2/proposals')))
proposal_id=select_proposal(PROPOSAL_ID)
payload=fetch(proposal_id,page_size=13)
display({'proposal_id':proposal_id,'ofertas':len(payload['offers']),'origens':sum(len(o['activities']) for o in payload['offers']),
         'schema':payload['schema_version'],'hash_proposta':payload['proposal_sha256'],'origem':'API HTTP → PostgreSQL'})
display(table(payload['offers']))


**Executar:** filtrar ciclo/escola/curso/matriz/coorte/período/turno/disciplina/tipo/status e abrir uma oferta. **Resultado:** cada filtro faz nova consulta HTTP; detalhe contém atividades, demanda e lineage.

In [ ]:

controls=explorer(proposal_id,payload)


**Evidência:** conferir P164 exclusivamente pela API. **Resultado:** uma oferta, três atividades; 41 planejados não são 82 estudantes.

In [ ]:

circuitos=fetch(proposal_id,{'course_code':'2248','discipline_code':'P164'})
assert len(circuitos['offers'])==1
offer=circuitos['offers'][0]
assert sorted(a['reference_planned_students'] for a in offer['activities'])==[20,21,41]
display(table(circuitos['offers']))
display(pd.DataFrame(offer['activities']))


**Onde conferir:** [MLflow](http://localhost:5000), [RustFS](http://localhost:9001), [API](http://localhost:8000/docs), [Logs](http://localhost:8088). Em falha: `docker compose logs --tail 100 api inference worker mlflow`.